## S09 LAB Exercises
#### Author: Sergio Arca Montenegro

##### Imports

In [69]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

##### Upload the datsets

In [70]:
# Uploading the datasets for the exercises
train_file = "sent_train.csv"
valid_file = "sent_valid.csv"

# Reading Datasets
df_train = pd.read_csv(train_file)
df_valid = pd.read_csv(valid_file)

# We looked at the first few rows to see the "structure"
print("Train Data:")
display(df_train.head())

print("Quantity of each emotion:")
print(df_train['label'].value_counts())

Train Data:


,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


Quantity of each emotion:
label
2    6178
1    1923
0    1442
Name: count, dtype: int64


##### Text processing - Tokenization

In [71]:
#Maximum number of words to keep in vocabulary
MAX_WORDS = 10000 

# Maximun length of sequency of tokens
MAX_SEQ_LENGTH = 50 

# We separate the texts and the labels
X_train_text = df_train['text'].values
y_train = df_train['label'].values
X_valid_text = df_valid['text'].values
y_valid = df_valid['label'].values

# We start the tokenizer and train it using only the training data
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

# Strings to integrer
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_valid_seq = tokenizer.texts_to_sequences(X_valid_text)

# We fill with 0s until we reach the maximum sequence to make them all "even"
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
X_valid_pad = pad_sequences(X_valid_seq, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')

##### LSTM Model

In [72]:
# Dimension size
dim = 128

# Creation of the sequential model
lstm_model = Sequential() 
lstm_model.add(Embedding(input_dim=MAX_WORDS, output_dim=dim, input_length=MAX_SEQ_LENGTH))
lstm_model.add(LSTM(64, return_sequences=False))
lstm_model.add(Dropout(0.5)) # We avoid overfitting
lstm_model.add(Dense(32, activation='relu'))
lstm_model.add(Dense(3, activation='softmax'))

# We compile the model and print a summary
lstm_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.build(input_shape=(None, MAX_SEQ_LENGTH)) # Fix to avoid getting "unbuild" in the summary 
lstm_model.summary()

C:\Users\sergi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 50, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,331,587 (5.08 MB)

 Trainable params: 1,331,587 (5.08 MB)

 Non-trainable params: 0 (0.00 B)

##### Model Training

In [73]:
EPOCHS = 10
BATCH_SIZE = 64

trained_model = lstm_model.fit(
    X_train_pad, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE,
    validation_data=(X_valid_pad, y_valid)
)

Epoch 1/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.6455 - loss: 0.9032 - val_accuracy: 0.6558 - val_loss: 0.8798
Epoch 2/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.6474 - loss: 0.8962 - val_accuracy: 0.6558 - val_loss: 0.8799
Epoch 3/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.6474 - loss: 0.8962 - val_accuracy: 0.6558 - val_loss: 0.8796
Epoch 4/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.6474 - loss: 0.8942 - val_accuracy: 0.6558 - val_loss: 0.8795
Epoch 5/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.6474 - loss: 0.8932 - val_accuracy: 0.6558 - val_loss: 0.8761
Epoch 6/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.6471 - loss: 0.8104 - val_accuracy: 0.6918 - val_loss: 0.7464
Epoch 7/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.6531 - loss: 0.7093 - val_accuracy: 0.6817 - val_loss: 0.7623
Epoch 8/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.7052 - loss: 0.6422 - val_accu